# External validation on PROSTATEx — frozen E1 model (Colab)

**Task D.** The frozen E1 model (epoch 88) run once over the 204-case PROSTATEx
cohort. **Inference only**: no training, no fine-tuning, no threshold search, no
model selection. Needs a GPU runtime.

### Read this before you look at the numbers

PROSTATEx is an **independent** cohort — different scanners, different
annotators, different acquisition geometry (0.3–0.6 mm in-plane against
Prostate158's ~0.47 mm; 3.0–4.5 mm through-plane). **Expect lower Dice than the
Prostate158 test set.** That is a finding about generalization, which is exactly
what JMIR asks for when it requires independent dataset validation. It is not a
failure, and this framing is fixed here, before the numbers exist, so it cannot
be rationalized afterwards.

### Upload checklist

Upload the cohort to Drive preserving the layout:

```
/content/drive/MyDrive/THESIS_PROSTATE158/dataset/external/prostatex/
├── raw/images/t2/          (204 files)
├── raw/masks/pz/           (204 files)
├── raw/masks/rest/         (204 files)
└── metadata/prostatex_manifest.csv
```

The manifest is **required**, not optional: 15 cases use non-canonical mask
filenames that cannot be derived from the case id. Section 5 refuses to continue
if any of these is short — a partial upload is the realistic failure here and it
would otherwise be scored as if the cohort were complete.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration (edit here)

In [ ]:
REPO_URL     = "https://github.com/amadeussandro/prostate-mri-segmentation-thesis.git"
REPO_BRANCH  = "main"
PROJECT_DIR  = "/content/prostate-mri-segmentation-thesis"

DRIVE_BASE   = "/content/drive/MyDrive/THESIS_PROSTATE158"
PROSTATEX_ROOT = f"{DRIVE_BASE}/dataset/external/prostatex"

# The frozen RQ1 model — identical weights to the Prostate158 test result.
CHECKPOINT_PATH = f"{DRIVE_BASE}/results/exp_e1_dice_ce/best_model.pt"
OUTPUT_DIR      = f"{DRIVE_BASE}/results/external_prostatex_e1"

EXPECTED_SHA256 = "2366e3c68f5ec616b19c0e7df39f9135898fd287b9436ce5dfab7baf8528d5d2"
EXPECTED_CASES  = 204

for k in ["PROJECT_DIR", "DRIVE_BASE", "PROSTATEX_ROOT", "CHECKPOINT_PATH", "OUTPUT_DIR"]:
    print(f"{k:16s} = {globals()[k]}")
print(f"{'expected SHA':16s} = {EXPECTED_SHA256}")

## 3. Repository setup (clone or pull — no duplicates)

In [ ]:
import os, subprocess

def sh(*args, check=True):
    print("$", " ".join(args))
    return subprocess.run(args, check=check)

if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    sh("git", "clone", "--branch", REPO_BRANCH, REPO_URL, PROJECT_DIR)
else:
    sh("git", "-C", PROJECT_DIR, "fetch", "origin", REPO_BRANCH)
    sh("git", "-C", PROJECT_DIR, "checkout", REPO_BRANCH)
    sh("git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", REPO_BRANCH)

os.chdir(PROJECT_DIR)
print("\ncwd:", os.getcwd())
sh("git", "-C", PROJECT_DIR, "log", "--oneline", "-1")

## 4. Install dependencies

In [ ]:
%pip install -q nibabel>=5.4 scipy>=1.13 pyyaml>=6.0 pandas>=2.0
import torch, nibabel, numpy, scipy
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print("nibabel", nibabel.__version__, "| numpy", numpy.__version__)
if not torch.cuda.is_available():
    print("\nWARNING: no GPU. 204 cases on CPU will be slow — "
          "Runtime > Change runtime type > GPU.")

## 5. Upload completeness gate

The realistic failure mode is a Drive upload that silently dropped files. 612
NIfTI volumes through the web uploader is exactly the situation where that
happens, and a short cohort scored as complete would produce wrong numbers with
no error. This cell refuses to continue unless all four pieces are present.

In [ ]:
import os, sys, csv

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "scripts"))

expected = {
    "raw/images/t2":   EXPECTED_CASES,
    "raw/masks/pz":    EXPECTED_CASES,
    "raw/masks/rest":  EXPECTED_CASES,
}
problems = []
for rel, n_expected in expected.items():
    d = os.path.join(PROSTATEX_ROOT, rel)
    if not os.path.isdir(d):
        problems.append(f"missing directory: {d}")
        continue
    n = len([f for f in os.listdir(d) if f.endswith(".nii.gz")])
    status = "OK" if n == n_expected else "SHORT"
    print(f"  {rel:18s} {n:4d} / {n_expected}   {status}")
    if n != n_expected:
        problems.append(f"{rel} holds {n} files, expected {n_expected}")

manifest = os.path.join(PROSTATEX_ROOT, "metadata", "prostatex_manifest.csv")
if not os.path.isfile(manifest):
    problems.append(f"manifest missing: {manifest} (REQUIRED — 15 cases have "
                    "non-canonical filenames that cannot be derived from the case id)")
else:
    rows = list(csv.DictReader(open(manifest, encoding="utf-8")))
    print(f"  {'manifest':18s} {len(rows):4d} rows")
    if len(rows) != EXPECTED_CASES:
        problems.append(f"manifest lists {len(rows)} cases, expected {EXPECTED_CASES}")

if problems:
    raise FileNotFoundError(
        "Cohort incomplete — a partial Drive upload is the likely cause. "
        "Re-upload the missing pieces before running anything:" +
        "".join(chr(10) + "  - " + p for p in problems))
print("\nOK — all 204 cases and the manifest are present.")

## 6. Checkpoint identity

A filename proves nothing; the hash does. These must be the same weights that
produced the Prostate158 test result, or the comparison between the two cohorts
means nothing.

In [ ]:
import hashlib, os

digest = hashlib.sha256()
with open(CHECKPOINT_PATH, "rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        digest.update(block)
sha = digest.hexdigest()

print(f"checkpoint : {CHECKPOINT_PATH}")
print(f"size       : {os.path.getsize(CHECKPOINT_PATH):,} bytes")
print(f"SHA-256    : {sha}")
assert sha == EXPECTED_SHA256, (
    "SHA-256 MISMATCH — this is not the frozen E1 checkpoint.\n"
    f"  expected {EXPECTED_SHA256}\n  got      {sha}")
print("\nOK — frozen E1 confirmed.")

## 7. Dry run: harmonize and verify, without the model

PROSTATEx ships PZ and "rest" as two separate binary masks; the model's
convention is `0=background, 1=CG, 2=PZ`. The combination is built here and
never written back into `raw/`.

Two details this step gets right, both of which would otherwise corrupt the run
silently: masks are combined with `mask > 0` rather than `mask == 1` (case
ProstateX-0086 carries a stray voxel of value 2 in its PZ mask), and zone
disjointness is re-measured per case instead of trusted.

Nothing here needs the GPU, so a failure costs you seconds rather than a run.

In [ ]:
# Colab expands {NAME} from the Python namespace inside a ! line.
!python scripts/run_prostatex_external_validation.py \
    --prostatex-root "{PROSTATEX_ROOT}" \
    --output-dir     "{OUTPUT_DIR}" \
    --checkpoint     "{CHECKPOINT_PATH}" \
    --prepare-only

## 8. Run the external validation

Inference over 204 cases through the **same** inference-and-reconstruction path
used for RQ1 and RQ2 (`predict_case_reconstructed`), so there is no second
implementation that could drift from the one the thesis reports.

In [ ]:
!python scripts/run_prostatex_external_validation.py \
    --prostatex-root "{PROSTATEX_ROOT}" \
    --checkpoint     "{CHECKPOINT_PATH}" \
    --output-dir     "{OUTPUT_DIR}"

## 9. Results

In [ ]:
import os
from IPython.display import Markdown, display
import pandas as pd

report = os.path.join(OUTPUT_DIR, "report.md")
if os.path.exists(report):
    display(Markdown(open(report, encoding="utf-8").read()))

geom = os.path.join(OUTPUT_DIR, "cohort_geometry.csv")
if os.path.exists(geom):
    df = pd.read_csv(geom)
    print("\nCases cropped by the 442 target:",
          int(df["inplane_exceeds_target"].sum()))
    lost = df[(df["class1_lost"] > 0) | (df["class2_lost"] > 0)]
    print("Cases losing annotated foreground to the crop:", len(lost))
    if len(lost):
        display(lost[["case_id", "shape", "class1_lost", "class2_lost"]])
    print("\nIn-plane scale vs Prostate158: "
          f"{df['scale_vs_prostate158'].min():.2f}x to {df['scale_vs_prostate158'].max():.2f}x")

## 10. How to write this up

Do **not** present these numbers as a like-for-like contrast with the
Prostate158 test result. The annotators, scanners, geometry and zone definitions
all differ, so the comparison is not controlled. They answer a different
question — does a model trained on one cohort retain useful zonal segmentation
on an independent one — and the honest unit of that answer is the **gap**,
reported next to the measured covariate shift in Section 3 of the report that
partly explains it.

The pipeline does no in-plane resampling, so part of any drop is scale shift
rather than model failure. If you want to separate the two, that is a second
run with a resampling adapter, and it is a scientific decision to take with your
supervisor rather than a knob to turn quietly.